### 1.1.2. Set Declaration vs. Definition
This notebook explains **how the set receives its members**.

| Form | Example | Meaning |
|---|---|---|
| Declaration | `set PROD;` | The set exists, but data is supplied later. |
| Definition | `set SHIFTS = {"Day", "Evening", "Night"};` | AMPL computes/fixes the set from an expression. |
| Data assignment | `set PROD := bands coils plate;`<br>  `set ORIG['bands'] := A, B, C;`| A data section supplies members for a declared set. |
| Default value | `set PROD default {};` <br> `set ORIG{PROD} default {};`| AMPL uses fallback data only if no data is supplied. |

#### The `=` Operator (Symbolic Definition)

When a set declaration includes `= SEXPR`, the set is **defined by an expression**.

```python
set SHIFTS = {"Day", "Evening", "Night"};
```
This means AMPL computes the set from the expression on the right side. A set defined with `=` is not meant to receive data later from a `.dat` file or from `let`.

Use `=` when the set is structural and should stay fixed for every dataset.
```python
set PROD ;
set ORIG{PROD};
set ORIG['bands'] = {"A", "B", "C"};
```
AMPL re-evaluates the expression whenever any set or parameter it depends on changes. Because the value is fixed by the model, a data section cannot re-assign it.

Symbolic definition is appropriate when:
- The set is a structural invariant of the model (e.g., shift categories, a range of periods).
- The set is derived from other sets or parameters and must track them automatically.
- The set is small and its members are unlikely to change across problem instances.

The expression on the right of `=` may be any valid set expression: a literal set, an arithmetic range, a filtered set comprehension, or a set-algebraic combination.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/Mihailby/AMPLKnowledge/blob/main/examples/1_1_2__1.ipynb)

In [ ]:
# Literal set — fixed members embedded in the model
set SHIFTS = {"Day", "Evening", "Night"};

set PLANTS;                                # Index set
set PROD1 {pl in PLANTS} =                 # Products by plant
    if pl = "P1" then {"bands", "coils", "plate"}
    else {"rods", "wire"};

# Arithmetic range parameterized by data — recomputed if start/end/interval change
param start integer;
param end > start integer;
param interval > 0 integer;
set YEARS = start .. end by interval;

# Filtered comprehension — CHEAP depends on PROD and cost; updated automatically
set PROD;
param cost {PROD} >= 0;
set CHEAP = {p in PROD: cost[p] < 10};

# set-algebraic combination
set PROD_A = {"bands", "coils"};
set PROD_B = {"coils", "plate", "rods"};

set ALL_PROD   = PROD_A union PROD_B;          # {"bands", "coils", "plate", "rods"}
set COMMON     = PROD_A inter PROD_B;          # {"coils"}
set ONLY_A     = PROD_A diff PROD_B;           # {"bands"}
set EXCLUSIVE  = PROD_A symdiff PROD_B;        # {"bands", "plate", "rods"}
set PAIRS      = PROD_A cross PROD_B;          # all ordered pairs (a, b)

#### The `:=` Operator (Data Assignment)

`:=` assigns explicit member values to a set that was declared without `=`. It appears in:
- A **data file** (inside a `data;` section), to load instance data.
- A **`let` command** in an AMPL script or interactive session, to override the current value at runtime.

Unlike the `=` phrase in a declaration, `:=` does not bind the set permanently. A later `let` or data reload can replace the assignment. Using `:=` inside a model declaration is formally deprecated; use `=` or `default` there instead.

The two fragments below show the same set populated first via a data file, then overridden with `let`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/Mihailby/AMPLKnowledge/blob/main/examples/1_1_2__2.ipynb)

In [ ]:
# --- model file ---
set PROD;
set PLANTS;                 # Index set

set PROD1 {PLANTS} ;        # Products by plant
    if pl = "P1" then {"bands", "coils", "plate"}
    else {"rods", "wire"};

# --- data file (.dat) ---
data;
set PROD := bands coils plate;

# --- interactive session or script
let PROD := {"bands", "coils", "plate", "rods"};  # Replace the set completely
let PROD := PROD union {"wire"};    # Add new members to existing set
let PROD := PROD diff {"bands"};    # Remove members

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
Using <code>:=</code> inside a model-file declaration (e.g., <code>set PROD := {"a", "b"};</code>) is accepted by current AMPL for historical compatibility but is deprecated. The intended meaning of <code>:=</code> in a declaration is identical to <code>=</code>. Write <code>=</code> in the model and reserve <code>:=</code> for data files and <code>let</code> commands.</div>

#### String Members: Quoting Rules

AMPL distinguishes two contexts for string literals: **model context** (inside `.mod` files and
`ampl.eval()` calls) and **data context** (inside `.dat` files and `data;` sections).

**Model context — quotes are always required.**
Any string literal inside a model expression must be enclosed in single quotes (`'A&P'`) or
double quotes (`"Bell+Howell"`). Both delimiters are equivalent; the choice is the modeler's.
A literal string that appears without quotes in a model expression is interpreted as a name
(a reference to a declared set, parameter, or other component), not as a string value.

**Data context — quotes are optional for simple strings.**
Inside a `data;` section or `.dat` file, AMPL allows the quotes around a string to be omitted
if the string consists only of characters that could appear in a name or number (letters, digits,
underscores, `.`, `+`, `-`), provided the string would not otherwise be parsed as a number.
The string `bands` may be written without quotes; the string `A&P` requires quotes because
`&` is not a valid name character.

**Case sensitivity.**
Upper-case and lower-case letters are always distinct in AMPL. The strings `"bands"`, `"Bands"`,
and `"BANDS"` denote three different set members in every context.

**Numbers vs. numeric-looking strings.**
The number `1` and the string `"1"` are distinct AMPL objects. A number is stored as a
floating-point value; different representations of the same number (`100` and `1E+2`) are the
same set member. A string that looks like a number (e.g., `"100"`) is a character sequence,
not a floating-point value, and cannot participate in arithmetic. In a data context, writing
`100` (without quotes) produces the number 100; writing `"100"` produces the string `"100"`.
Mixing numbers and strings in the same set is syntactically valid but almost always a modelling
error.

In [ ]:
# --- MODEL CONTEXT: quotes are always required for string literals ---

# Correct: both quote styles are accepted
set SHIFTS = {"Day", "Evening", "Night"};               # double quotes
set SEASONS = {'Spring', 'Summer', 'Autumn', 'Winter'}; # single quotes

# Correct: string containing special characters must be quoted
set CODES = {'A&P', '23RPFG', '486/33C', 'New_York'};

# Membership test with a string literal — quotes required in model context
param has_day = if "Day" in SHIFTS then 1 else 0;

# Number 1 and string "1" are distinct
set MIXED = {1, "1"};        # two distinct members: numeric 1 and string "1"
display card(MIXED);         # 2 — not 1

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
A string literal in <strong>model context</strong> (inside <code>.mod</code> files and
<code>ampl.eval()</code> calls) must always be quoted, even if it consists only of letters.
Writing <code>"Day" in SHIFTS</code> is correct; writing <code>Day in SHIFTS</code> is a
reference to a declared set named <code>Day</code>, not to the string <code>"Day"</code>.</div>

#### Data Context: Quoting Rules in `.dat` Files

In a `data;` section or `.dat` file the rules relax slightly. Quoting remains mandatory for
strings that contain characters outside the set `[A-Za-z0-9_.+-]`. For alphanumeric strings
(letters, digits, underscores) that would not be interpreted as numbers, quotes are optional.

The table below summarises when quotes are needed.

| String value | In model context | In data context | Reason |
|---|---|---|---|
| `bands` | `"bands"` required | `bands` or `"bands"` | Alphanumeric — quotes optional in data |
| `New_York` | `"New_York"` required | `New_York` or `"New_York"` | Underscore is a valid name character |
| `A&P` | `"A&P"` required | `"A&P"` required | `&` is not a valid name character |
| `27sep` | `"27sep"` required | `27sep` or `"27sep"` | Starts with digit; not a legal number |
| `100` | `"100"` for the string; `100` for the number | `"100"` for the string; `100` for the number | Distinguish string from number |
| `Gary, Indiana` | `"Gary, Indiana"` required | `"Gary, Indiana"` required | Contains space and comma |

#### When to Define in the Model vs. a Data File

The choice between embedding set content in the model (`=`) versus supplying it via a data file (`:=`) turns on whether the set is **structural** or **instance-varying**.

| Criterion | Use `=` in model | Use `:=` in data file |
|---|---|---|
| Set is small (a handful of elements) | Yes | Possible but unnecessary |
| Set is a fundamental model invariant | Yes | No — data can inadvertently alter it |
| Set varies across problem instances | No | Yes |
| Set has hundreds or thousands of members | No | Yes |
| Model is reused with different datasets | No | Yes — swap data file, not model |

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
The canonical AMPL discipline is: declare sets in the model, supply their data from outside. This keeps the model file stable and the data file the only thing that changes between runs.</div>


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/Mihailby/AMPLKnowledge/blob/main/examples/1_1_2__3.ipynb)

In [ ]:
# Model file — structural, fixed: defined with =
set SHIFTS = {"Day", "Evening", "Night"};  # invariant across all instances

# Model file — instance-varying: declared only
set PROD;    # membership comes from the data file
set CITIES;  # membership comes from the data file

set PROD1 {pl in PLANTS} =   # Defined indexed set: computed in the model             
    if pl = "P1" then {"bands", "coils", "plate"}
    else {"rods", "wire"};   # Products by plant

#### Set Data Assignment in amplpy:

| # | Pattern | Syntax | Notes |
|---|---------|--------|-------|
| 1 | Raw AMPL | `ampl.eval("set PROD := {'bands', 'coils', 'plate'};")` | Must include `:=` and proper AMPL syntax |
| 2 | Property assignment | `ampl.set["PROD"] = ["bands", "coils", "plate"]` | Set must be declared first |
| 3 | Method assignment | `ampl.get_set("PROD").set_values(["bands", "coils", "plate"])` | Set must be declared first |
| 4 | DataFrame assignment | `ampl.set_data(df, 'PROD')` | Set must be declared first; can include parameters |


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/Mihailby/AMPLKnowledge/blob/main/examples/1_1_2__4.ipynb)

In [ ]:
ampl.reset()

# Pattern 1: Raw AMPL - simplest for static data
ampl.eval("set PROD1 := {'bands', 'coils', 'plate'};")

# Pattern 2: Property assignment - most Pythonic
# 🏆 RECOMMENDED: Use this pattern for 80% of cases
ampl.eval("set PROD2;")
ampl.set["PROD2"] = ["bands", "coils", "plate"]

# Pattern 2 variant: From dictionary keys
ampl.eval("set PROD3;")
bounds = {"bands": (0, 100), "coils": (0, 100), "plate": (0, 100)}
ampl.set["PROD3"] = list(bounds.keys())

# Pattern 3: Method assignment - explicit and chainable
ampl.eval("set PROD4;")
ampl.get_set("PROD4").set_values(["bands", "coils", "plate"])

# Pattern 4: DataFrame with tolist() - good for pandas integration
# 🏆 Use this pattern when data comes from files or has parameters
ampl.eval("set PROD5;")
df = pd.DataFrame(["bands", "coils", "plate"], columns=["PROD5"])
ampl.get_set("PROD5").set_values(df["PROD5"].tolist())

#### ⚠️ Edge case: Attempting to assign data to a set defined with `=`

A set declared with `=` is permanently bound to its expression. Any attempt to supply its value through a data section or `let` command will produce an error. This is a common mistake when converting a model from one that originally defined the set in the data to one with an embedded definition.

In [ ]:
# Model file — set defined with =
set SHIFTS = {"Day", "Evening", "Night"};
    
# Correct fix: use 'default' if override is needed
set SHIFTS default {"Day", "Evening", "Night"};

# ❌[ERROR] 
	# ...
	# SHIFTS is already defined
	# context:  set  >>> SHIFTS  <<< default {"Day", "Evening", "Night"};

#### Members: strings, numbers
A set's members may be of two kinds:

| Kind | Example members | Notes |
|---|---|---|
| **Symbolic (string)** | `"bands"`, `"GARY"`, `"Day"` | Enclosed in quotes in model context; quotes are optional for alphanumeric strings in data context |
| **Numeric** | `1`, `2.5`, `2020` | Stored as floating-point; `100` and `1E+2` are the same member |

The number `1` and the string `"1"` are **distinct** objects in AMPL. Mixing types in one set
is syntactically valid but is almost always a modelling error.

String literals in **model context** (inside `.mod` files and `ampl.eval()` calls) must always be
quoted вЂ” either single quotes (`'bands'`) or double quotes (`"bands"`). In **data context**
(inside `.dat` files and `data;` sections) quotes are optional for simple alphanumeric strings.

#### ⚠️ Edge case: Numeric set vs. string set type confusion

The number `100` and the string `"100"` are **different** AMPL objects with different types.
Their intersection is empty. In a data context, writing `100` (without quotes) produces the
number 100; writing `"100"` produces the string `"100"`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/Mihailby/AMPLKnowledge/blob/main/examples/1_1_2__5.ipynb)

In [ ]:
set ORDER_IDS_NUM = {100, 200, 300};        # Numeric members
set ORDER_IDS_STR = {"100", "200", "300"};  # String members that look like numbers

# Cardinality of each: 3
display card(ORDER_IDS_NUM);   # 3
display card(ORDER_IDS_STR);   # 3

# Intersection is EMPTY: number 100 is not the string "100"
display ORDER_IDS_NUM inter ORDER_IDS_STR;   # {}

#### See also

- [§1.1 — Sets](<1_1 Sets.ipynb>) — General set syntax and `SEXPR`.
- [§1.1.1 — Flat vs Indexed Sets](<1_1_1 Flat vs Indexed Sets.ipynb>) — Difference between one set and a family of sets.
- [§1.1.3 — Subsets and Supersets](<1_1_3 Subsets and Supersets.ipynb>) — More detail on `within SEXPR`.
- [§1.1.4 — Unordered Sets](<1_1_4 Unordered Sets.ipynb>) — More detail on `ordered` and `circular`.
